<a href="https://colab.research.google.com/github/hemanthvnp/MvDA_Advancement/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MvDA on Colab

Multi-view Discriminant Analysis, reading **ColorFERET** straight from Google Drive (no manual download). If you upload a zip instead of using GitHub, skip cell 1 and `cd` into the folder.

## 1. Get the code

In [ ]:
!git clone https://github.com/hemanthvnp/MvDA_Advancement.git mvda
%cd mvda
!pip -q install -r requirements.txt

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Locate ColorFERET and confirm it has images

If the image count is 0, your Drive copy is metadata-only and you need the real `.ppm`/`.ppm.bz2` files.

In [ ]:
!find /content/drive/MyDrive -maxdepth 4 -iname '*colorferet*' -type d 2>/dev/null | head
FERET_ROOT = '/content/drive/MyDrive/colorferet'  # <-- set to the path printed above
!echo 'image files found:'; find "$FERET_ROOT" -type f \( -iname '*.ppm' -o -iname '*.ppm.bz2' -o -iname '*.png' -o -iname '*.jpg' \) 2>/dev/null | wc -l

## 4. Sanity check on UCI Multiple Features (auto-downloads, no Drive needed)

In [ ]:
!python experiments/run_mvda.py --dataset mfeat --mode concat --classifier ensemble

## 5. ColorFERET, MvDA-paper protocol (subject-disjoint, 7 poses)

Kan et al. 2016: train on the first 231 identities (4 images/pose), then **gallery/probe recognition on the remaining, unseen identities** across 7 poses. First run reads + caches all images (slow); later runs reuse the cache.

In [ ]:
!python experiments/run_feret.py --protocol disjoint --solver ratio \
    --feret-root "$FERET_ROOT" --feret-poses pl hl ql fa qr hr pr \
    --train-subjects 231 --images-per-pose 4 --gallery-pose fa \
    --feret-size 64 64 --pca 120 --save feret_paper_ratio.json

## 6. Does MvDA improve with the paper-based solvers?

Same paper protocol and split, swapping the discriminant solver:
- **`ratio`** — classical LDA (baseline).
- **`exponential`** — Exponential DA (robust to small-sample singularity, enlarges margins).
- **`harmonic`** — Harmonic-mean LDA (emphasizes confusable identity pairs).

Higher rank-1 accuracy for `exponential`/`harmonic` ⇒ the improvement holds on FERET.

In [ ]:
for solver in ['ratio', 'exponential', 'harmonic']:
    print('\n================', solver, '================')
    !python experiments/run_feret.py --protocol disjoint --solver $solver \
        --feret-root "$FERET_ROOT" --feret-poses pl hl ql fa qr hr pr \
        --train-subjects 231 --images-per-pose 4 --gallery-pose fa \
        --feret-size 64 64 --pca 120

## 7. (optional) Closed-set protocol

A simpler protocol: per-view split of all subjects, classify held-out single-pose images by nearest class mean.

In [ ]:
!python experiments/run_feret.py --protocol closed \
    --feret-root "$FERET_ROOT" --feret-poses fa fb hl hr --feret-size 64 64 --pca 120 --solver harmonic